# Lecture 4: Building and Manipulating Structures

## Overview
**Questions**
- How can I build molecules and bulk structures efficiently?
- How can I create supercells and point defects?
- How do I build surfaces and interfaces?

**Objectives**
- Use `ase.build` to construct common crystal structures
- Create supercells for defect calculations
- Introduce a point defect (substitutional impurity or vacancy)


## `ase.build` — tools for constructing structures

ASE provides a rich set of structure-building utilities in `ase.build`:

| Function | Purpose |
|----------|---------|
| `bulk` | Common crystal structures (FCC, BCC, diamond, wurtzite, ...) |
| `molecule` | Molecules from the G2 database |
| `surface` | Cut surfaces from a bulk structure |
| `add_adsorbate` | Add a molecule to a surface |
| `stack` | Stack two slabs |
| `make_supercell` | Create a supercell with an arbitrary transformation matrix |



In [ ]:
from ase.build import bulk, molecule, surface, make_supercell
from ase.visualize.plot import plot_atoms
import matplotlib.pyplot as plt
import numpy as np

# -- Common bulk structures --
diamond_C  = bulk('C',   'diamond',  a=3.57)
wurtzite_GaN = bulk('GaN', 'wurtzite', a=3.19, c=5.19)
rocksalt_MgO = bulk('MgO', 'rocksalt', a=4.21)

for name, atoms in [('Diamond C', diamond_C),
                    ('Wurtzite GaN', wurtzite_GaN),
                    ('Rocksalt MgO', rocksalt_MgO)]:
    print(f"{name:20s}: {atoms.get_chemical_formula():8s} "
          f"({len(atoms)} atoms, volume = {atoms.get_volume():.2f} Å³)")


## Supercells

For defect calculations — critical in quantum optics, where we want to model isolated point defects — we need a **supercell** large enough that the defect does not interact with its periodic images.

A rule of thumb is to use a supercell where the defect–defect image distance exceeds ~10 Å. For diamond (a = 3.57 Å) this means at least a 3×3×3 supercell.


In [ ]:
# Create a 3x3x3 supercell of diamond
T = np.diag([3, 3, 3])   # transformation matrix
diamond_super = make_supercell(diamond_C, T)

print(f"Primitive cell: {len(diamond_C)} atoms")
print(f"3×3×3 supercell: {len(diamond_super)} atoms")
print(f"Supercell dimensions: {diamond_super.cell.lengths()} Å")

# Check minimum periodic image distance
from ase.geometry import get_distances
_, dists = get_distances(diamond_super.positions[[0]],
                         diamond_super.positions[[0]],
                         cell=diamond_super.cell, pbc=True)
# Remove self-distance
print(f"Cell edge length: {diamond_super.cell[0,0]:.2f} Å")


## Creating Point Defects

Point defects are central to quantum optics: the NV centre in diamond, the boron vacancy in hBN, and rare-earth substitutionals in wide-gap semiconductors are all point defects that emit single photons.

### The NV centre in diamond

The **nitrogen-vacancy (NV) centre** consists of a substitutional nitrogen atom (N_C) adjacent to a carbon vacancy (V_C). It is the best-studied solid-state qubit and single-photon emitter.

```{note}
The NV⁻ (negatively charged) state is the most useful for quantum optics applications. Creating the correct charge state in DFT requires careful treatment of the Fermi level.
```


In [ ]:
from ase import Atoms

# Build a 3x3x3 diamond supercell
T = np.diag([3, 3, 3])
nv_cell = make_supercell(diamond_C, T)
print(f"Pristine supercell: {len(nv_cell)} atoms of C")

# Step 1: Find two nearest-neighbour C atoms
from ase.geometry import get_distances
_, dist_matrix = get_distances(nv_cell.positions, nv_cell.positions,
                                cell=nv_cell.cell, pbc=True)
np.fill_diagonal(dist_matrix, np.inf)
nearest_idx = np.argmin(dist_matrix[0])   # nearest neighbour to atom 0
print(f"Nearest neighbour to atom 0: atom {nearest_idx}, "
      f"distance = {dist_matrix[0, nearest_idx]:.3f} Å")

# Step 2: Substitute atom 0 with nitrogen
symbols = list(nv_cell.get_chemical_symbols())
symbols[0] = 'N'
nv_cell.set_chemical_symbols(symbols)
print(f"After N substitution: {nv_cell.get_chemical_formula()}")

# Step 3: Delete the nearest C neighbour (create vacancy)
del nv_cell[nearest_idx]
print(f"After vacancy creation: {nv_cell.get_chemical_formula()}")
print(f"NV supercell: {len(nv_cell)} atoms")


In [ ]:
# Visualise the NV centre region
fig, ax = plt.subplots(figsize=(6, 6))
plot_atoms(nv_cell, ax, rotation=('10x,10y,0z'), radii=0.4)
ax.set_title('NV centre in diamond supercell
(N in blue, C vacancy at origin)')
ax.axis('off')
plt.tight_layout()
plt.show()


## Building Surfaces

Surfaces are important for photonic devices and for understanding how quantum emitters interact with their environment.


In [ ]:
from ase.build import surface as build_surface

# Diamond (100) surface — relevant for NV photonics applications
diamond_100 = build_surface(diamond_C, (1, 0, 0), layers=6, vacuum=10.0)
print(f"Diamond (100) slab: {len(diamond_100)} atoms")
print(f"Cell dimensions: {diamond_100.cell.lengths()} Å")
print(f"  (z includes 10 Å vacuum)")

fig, ax = plt.subplots(figsize=(4, 7))
plot_atoms(diamond_100, ax, rotation=('0x,0y,0z'), radii=0.4)
ax.set_title('Diamond (100) slab')
ax.axis('off')
plt.tight_layout()
plt.show()


## Key Points

- `ase.build.bulk` generates common crystal structures from minimal input
- Supercells are created with `make_supercell` using a 3×3 transformation matrix
- Point defects are introduced by modifying symbols (substitution) or deleting atoms (vacancy)
- For quantum optics applications, defect supercells typically need 100–200+ atoms

## Exercise 4.1

Build a 4×4×1 supercell of hexagonal boron nitride (hBN). hBN is a 2D material, so use `pbc=[True, True, False]` and add 20 Å of vacuum in the z-direction. Print the supercell dimensions and number of atoms.

## Exercise 4.2 (Research-level)

The **boron vacancy** (V_B) in hBN is an emerging single-photon emitter. Starting from your hBN supercell, create a V_B defect by deleting one boron atom. What fraction of atoms does the defect concentration correspond to? Is this supercell large enough for an isolated defect calculation?
